# Politically Motivated Crime in Germany: Right-Wing Offences and AfD Election Results

### Part 1: Data Preparation. Police statistics (PMK) and official election data, 2014 to 2025

This notebook prepares the data for the analysis in `02_analysis.ipynb`. It loads the raw files, checks them and saves clean tables. It contains no analysis and no charts.

Four tables are prepared:

1. **AfD election results:** share of votes for the AfD in each federal state in six elections, 2014 to 2025.
2. **Population:** inhabitants of each federal state on 31 December, 2022 to 2025.
3. **Right-wing offences, federal level:** registered offences 2014 to 2025, in total and by type of offence.
4. **Right-wing offences, federal states:** registered offences 2022 to 2025, in total and per 100,000 inhabitants.

Which offences count as right-wing motivated is not defined in this project. The classification of the police is used (phenomenon area "PMK -rechts-").

Python is used for loading, checking and saving.

| Area | Scope |
|---|---|
| Subject | Politically motivated crime, phenomenon area right-wing (PMK -rechts-), as recorded by the police and published by the BKA and BMI |
| Period | 2014 to 2025 at federal level; 2022 to 2025 for the federal states |
| Crime data | Figures transcribed from the published PDF reports, each with file name and page as source |
| Election data | Federal elections 2017, 2021 and 2025; European elections 2014, 2019 and 2024 (Federal Returning Officer) |
| Population data | Destatis table 12411-0010, reference date 31 December |
| Measures | Registered offences, violent offences, offences per 100,000 inhabitants, AfD share of votes in % |
| Output | Four clean tables in `data/clean/` |

German terms used in this notebook:

| German term | Term in this notebook |
|---|---|
| Politisch motivierte Kriminalität (PMK) | Politically motivated crime |
| PMK -rechts- | Right-wing motivated offences |
| Gewaltdelikte | Violent offences |
| Bundestagswahl (BTW) | Federal election |
| Europawahl (EW) | European election |
| Zweitstimme | Second vote: the vote for a party list in a federal election |
| Bundesland | Federal state |
| Drucksache | Printed paper of the German Bundestag |

## 1. Import Libraries

In [1]:
from pathlib import Path

import pandas as pd

## 2. Define Parameters and File Paths

The folder `data/raw` is searched upwards from the notebook folder, so the notebook runs from any subfolder of the project.

The state tables start in 2022. For earlier years no figures per federal state were found in the published sources (see Section 10).

In [2]:
# Locate data/raw by walking up from the notebook folder
start = Path.cwd()
DATA_RAW = next(p / "data" / "raw" for p in [start, *start.parents] if (p / "data" / "raw").exists())
DATA_CLEAN = DATA_RAW.parent / "clean"
DATA_CLEAN.mkdir(exist_ok=True)

ELECTIONS = DATA_RAW / "wahlen"
POPULATION_CSV = DATA_RAW / "bevoelkerung" / "12411-0010_de.csv"       # Destatis, 31 December
PMK_FEDERAL_CSV = DATA_CLEAN / "pmk_rechts_bund.csv"                   # transcribed from the PDF reports
PMK_STATES_CSV = DATA_CLEAN / "pmk_rechts_laender.csv"                 # transcribed from the Bundestag papers

# Election files in flat format: file, election, year, number of the vote that counts
# Federal election: second vote = 2. European election: only one vote = 1.
ELECTION_FILES = [
    (ELECTIONS / "btw17_kerg2.csv", "federal", 2017, 2),
    (ELECTIONS / "btw21_kerg2.csv", "federal", 2021, 2),
    (ELECTIONS / "btw25_kerg2.csv", "federal", 2025, 2),
    (ELECTIONS / "ew_2014_2019_2024" / "ew2019_kerg2.csv", "european", 2019, 1),
    (ELECTIONS / "ew_2014_2019_2024" / "ew2024_kerg2.csv", "european", 2024, 1),
]
EW_2014_CSV = ELECTIONS / "ew_2014_2019_2024" / "ew2014_kerg.csv"       # older table format

PARTY = "AfD"
FIRST_YEAR_STATES, LAST_YEAR = 2022, 2025

# English names of the federal states (states missing here have the same name in both languages)
STATE_NAMES_EN = {
    "Bayern": "Bavaria", "Hessen": "Hesse", "Mecklenburg-Vorpommern": "Mecklenburg-Western Pomerania",
    "Niedersachsen": "Lower Saxony", "Nordrhein-Westfalen": "North Rhine-Westphalia",
    "Rheinland-Pfalz": "Rhineland-Palatinate", "Sachsen": "Saxony", "Sachsen-Anhalt": "Saxony-Anhalt",
    "Thüringen": "Thuringia", "Bundesgebiet": "Germany",
}

# English names of the offence types in the transcribed tables
OFFENCE_NAMES_EN = {
    "gesamt": "total", "gewalt": "violent", "propaganda": "propaganda",
    "volksverhetzung": "incitement_to_hatred", "sachbeschaedigung": "property_damage",
    "beleidigung": "insult", "noetigung_bedrohung": "coercion_threat",
}

## 3. Load the Data

Nine files are loaded: six election files, one population table and two tables with crime figures.

- **Election files 2017 to 2025** have one row per area, party and vote. Nine title rows stand above the header.
- **European election 2014** has an older format: one row per district, one column pair per party and a header that spans two rows. It contains votes but no percentages.
- **Population table** has one column per reference date. Every second column holds only a quality flag and is dropped.
- **Crime tables** were transcribed by hand from the PDF reports, because the tables in the PDFs cannot be read reliably by code. Every value has the file name and page of its source.

| Column | Meaning |
|---|---|
| `Gebietsart`, `Gebietsname` | Level (`Bund`, `Land`, district) and name of the area (election files) |
| `Gruppenname`, `Stimme` | Party and number of the vote (election files) |
| `Anzahl`, `Prozent` | Votes and share of valid votes (election files) |
| `year`, `offence`, `cases` | Reporting year, type of offence and registered offences (crime tables) |
| `state` | Federal state (crime table for the states) |
| `method` | `abgelesen` = value printed in the source, `gezaehlt` = counted from a list of cases in the source |
| `source` | File name and page of the PDF the value was taken from |

In [3]:
# Election files 2017 to 2025, flat format
elections = {
    (election, year): pd.read_csv(path, sep=";", skiprows=9, decimal=",", encoding="utf-8-sig")
    for path, election, year, vote in ELECTION_FILES
}

# European election 2014: header in two rows, Latin-1
ew_2014 = pd.read_csv(EW_2014_CSV, sep=";", skiprows=5, header=[0, 1], encoding="latin-1")

# Population: the header with the reference dates is in row 6, followed by the 16 states
population_raw = pd.read_csv(POPULATION_CSV, sep=";", skiprows=5, nrows=16, encoding="utf-8-sig")
population_raw = population_raw.rename(columns={population_raw.columns[0]: "state"})
date_columns = [c for c in population_raw.columns if c.startswith("31.12.") and len(c) == 10]
population_raw = population_raw[["state"] + date_columns]

# Crime tables, transcribed from the PDF reports
pmk_federal = pd.read_csv(PMK_FEDERAL_CSV).rename(
    columns={"jahr": "year", "deliktart": "offence", "anzahl": "cases"})
pmk_states = pd.read_csv(PMK_STATES_CSV).rename(
    columns={"jahr": "year", "gebiet": "state", "kennzahl": "offence", "anzahl": "cases", "methode": "method"})
for df in (pmk_federal, pmk_states):
    df["offence"] = df["offence"].map(OFFENCE_NAMES_EN)

tables = {"population_raw": population_raw, "pmk_federal": pmk_federal, "pmk_states": pmk_states}

## 4. Check Data Quality

The general check covers table size, missing values, duplicate rows and text in number columns. The election files are checked separately, because most of their rows and columns are not needed here.

In [4]:
# One summary row per table
number_columns = {"population_raw": date_columns, "pmk_federal": ["cases"], "pmk_states": ["cases"]}
quality = pd.DataFrame({
    name: {
        "rows": len(df),
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        # values that are present but cannot be read as a number
        "non_numeric_values": int(sum(
            (pd.to_numeric(df[c], errors="coerce").isna() & df[c].notna()).sum()
            for c in number_columns[name])),
    }
    for name, df in tables.items()
}).T
quality

,rows,columns,missing_values,duplicate_rows,non_numeric_values
population_raw,16,11,0,0,0
pmk_federal,69,4,0,0,0
pmk_states,128,6,0,0,0


In [5]:
# Election files: rows of the party at state and federal level, and missing values in these rows
vote_number = {(election, year): vote for path, election, year, vote in ELECTION_FILES}
rows = {}
for key, df in elections.items():
    party = df[(df["Gruppenname"] == PARTY) & (df["Stimme"] == vote_number[key])
               & df["Gebietsart"].isin(["Land", "Bund"])]
    rows[key] = {"rows_in_file": len(df), "party_rows": len(party),
                 "missing_votes": int(party["Anzahl"].isna().sum()),
                 "missing_share": int(party["Prozent"].isna().sum())}
pd.DataFrame(rows).T

rows_in_file  party_rows  missing_votes  missing_share
federal  2017         16137          17              0              0
         2021         18818          17              0              0
         2025         15617          17              0              0
european 2019         18811          17              0              0
         2024         16264          17              0              0

The three tables have no missing values, no duplicate rows and no text in number columns. Each of the five election files has 17 rows for the AfD at state and federal level (16 federal states and Germany), with votes and share in every row.

In [6]:
# First rows of each table
for name, df in tables.items():
    print(name)
    display(df.head(3))

population_raw


,state,31.12.2016,31.12.2017,31.12.2018,31.12.2019,31.12.2020,31.12.2021,31.12.2022,31.12.2023,31.12.2024,31.12.2025
0,Baden-Württemberg,10951893,11023425,11069533,11100394,11103043,11124642,11167721,11230740,11245898,11238761
1,Bayern,12930751,12997204,13076721,13124737,13140183,13176989,13105221,13176426,13248928,13245503
2,Berlin,3574830,3613495,3644826,3669491,3664088,3677472,3632853,3662381,3685265,3700577


pmk_federal


,year,offence,cases,source
0,2014,total,17020,"1805758.pdf, PDF-Seite 4"
1,2014,violent,1029,"1805758.pdf, PDF-Seite 4"
2,2014,propaganda,11071,"1805758.pdf, PDF-Seite 8"


pmk_states


,year,state,offence,cases,method,source
0,2022,Baden-Württemberg,total,1459,abgelesen,"2007594.pdf, PDF-Seite 8"
1,2022,Baden-Württemberg,violent,36,abgelesen,"2007594.pdf, PDF-Seite 8"
2,2022,Bayern,total,2541,abgelesen,"2007594.pdf, PDF-Seite 8"


## 5. Check Coverage: Years, Federal States and Offence Types

Before any calculation the tables are checked against what the analysis needs: the federal crime table must cover 2014 to 2025, the state table must contain all 16 federal states in every year, the federal states must have the same names in all sources, and the figures of the states must add up to the federal figure.

In [7]:
# Federal table: which offence types are available in which year? (1 = value available)
coverage = (pmk_federal.pivot(index="year", columns="offence", values="cases").notna().astype(int)
            [list(OFFENCE_NAMES_EN.values())])
coverage

offence,total,violent,propaganda,incitement_to_hatred,property_damage,insult,coercion_threat
year,,,,,,,
2014,1,1,1,1,1,0,1
2015,1,1,0,0,0,0,0
2016,1,1,0,0,0,0,0
2017,1,1,1,1,1,0,1
2018,1,1,1,1,1,0,1
2019,1,1,1,1,1,1,1
2020,1,1,1,1,1,1,0
2021,1,1,1,1,1,1,0
2022,1,1,1,1,1,1,1


In [8]:
print("State table: years", pmk_states["year"].min(), "to", pmk_states["year"].max(),
      "|", pmk_states["state"].nunique(), "states",
      "| duplicate year-state-offence rows:", pmk_states.duplicated(["year", "state", "offence"]).sum())
print("Rows per year:", pmk_states.groupby("year").size().to_dict())
print("Values by method:", pmk_states["method"].value_counts().to_dict())

# Do the 16 federal states have the same names in all sources?
names_crime = set(pmk_states["state"])
names_population = set(population_raw["state"])
names_election = set(elections[("federal", 2025)].query("Gebietsart == 'Land'")["Gebietsname"])
print("Same state names in all sources:", names_crime == names_population == names_election)

# Do the states add up to the federal figure?
state_sum = pmk_states.pivot_table(index="year", columns="offence", values="cases", aggfunc="sum")
federal_value = (pmk_federal[pmk_federal["offence"].isin(["total", "violent"])]
                 .pivot(index="year", columns="offence", values="cases").loc[state_sum.index])
state_sum.join(federal_value, lsuffix="_sum_of_states", rsuffix="_federal")

State table: years 2022 to 2025 | 16 states | duplicate year-state-offence rows: 0
Rows per year: {2022: 32, 2023: 32, 2024: 32, 2025: 32}
Values by method: {'abgelesen': 112, 'gezaehlt': 16}
Same state names in all sources: True


offence,total_sum_of_states,violent_sum_of_states,total_federal,violent_federal
year,,,,
2022,23493,1170,23493,1170
2023,28945,1270,28945,1270
2024,42788,1488,42788,1488
2025,42544,1598,42544,1598


The state table is complete: 16 federal states, four years, two values per state and year. The names of the federal states are identical in all sources, and the sum of the 16 states equals the federal figure in every year, for all offences and for violent offences.

The federal table has gaps, which are documented in Section 10:

- **Total and violent offences** are available for all twelve years.
- **Types of offence** are missing for 2015 and 2016.
- **Insult** is available from 2019 only.
- **Coercion and threat** is missing for 2020 and 2021.

16 of the 128 values in the state table were counted, not read: the violent offences per state in 2025 (Section 10).

## 6. AfD Election Results per Federal State

For the federal elections the second vote is used, for the European elections the only vote. The files of 2017 to 2025 already contain the share of valid votes. For the European election of 2014 the share is calculated as AfD votes divided by valid votes. In that file the rows of the federal states have the numbers 901 to 916 and the row for Germany the number 999.

In [9]:
parts = []
for (election, year), df in elections.items():
    party = df[(df["Gruppenname"] == PARTY) & (df["Stimme"] == vote_number[(election, year)])
               & df["Gebietsart"].isin(["Land", "Bund"])]
    parts.append(pd.DataFrame({
        "election": election,
        "year": year,
        "state": party["Gebietsname"],
        "afd_votes": party["Anzahl"].astype(int),
        "afd_share_pct": party["Prozent"],
    }))

# European election 2014: keep the rows of the 16 states and of Germany
number = ew_2014[ew_2014.columns[0]]
rows_2014 = ew_2014[number.between(901, 916) | (number == 999)]
parts.append(pd.DataFrame({
    "election": "european",
    "year": 2014,
    "state": rows_2014[rows_2014.columns[1]],
    "afd_votes": rows_2014[(PARTY, "Endgültig")].astype(int),
    "afd_share_pct": 100 * rows_2014[(PARTY, "Endgültig")] / rows_2014[("Gültige", "Endgültig")],
}))

afd_results = pd.concat(parts, ignore_index=True)
afd_results["state"] = afd_results["state"].replace(STATE_NAMES_EN)
afd_results["afd_share_pct"] = afd_results["afd_share_pct"].round(2)
afd_results = afd_results.sort_values(["year", "state"]).reset_index(drop=True)

# Check: 17 rows per election, and the votes of the 16 states add up to the votes in Germany
check = afd_results.groupby(["year", "election"]).agg(rows=("state", "size"))
check["votes_sum_of_states"] = afd_results[afd_results["state"] != "Germany"].groupby(["year", "election"])["afd_votes"].sum()
check["votes_germany"] = afd_results[afd_results["state"] == "Germany"].set_index(["year", "election"])["afd_votes"]
check

,,rows,votes_sum_of_states,votes_germany
year,election,,,
2014,european,17,2070014,2070014
2017,federal,17,5878115,5878115
2019,european,17,4104453,4104453
2021,federal,17,4803902,4803902
2024,european,17,6325890,6325890
2025,federal,17,10328780,10328780


In [10]:
# AfD share in % per state and election
afd_results.pivot(index="state", columns="year", values="afd_share_pct")

year,2014,2017,2019,2021,2024,2025
state,,,,,,
Baden-Württemberg,7.88,12.19,10.00,9.60,14.73,19.79
Bavaria,8.06,12.39,8.51,8.98,12.64,19.01
Berlin,7.91,12.05,9.90,8.41,11.60,15.23
Brandenburg,8.49,20.20,19.91,18.14,27.52,32.49
Bremen,5.79,10.00,7.68,6.88,10.23,15.07
Germany,7.05,12.64,10.98,10.34,15.89,20.80
Hamburg,6.01,7.82,6.49,5.03,8.01,10.87
Hesse,9.14,11.91,9.90,8.82,13.61,17.78
Lower Saxony,5.38,9.09,7.95,7.44,13.24,17.84


The table has 102 rows: 17 per election. In all six elections the votes of the 16 federal states add up exactly to the votes in Germany, so no state is missing or counted twice.

The two types of election are not directly comparable: turnout and the set of parties differ. The table keeps the type of election in the column `election`.

## 7. Population of the Federal States, 2022 to 2025

The population table is reshaped from one column per reference date to one row per state and year. Only the years of the state crime table are kept. From 2022 the figures are based on the census of 2022.

In [11]:
population = population_raw.melt(id_vars="state", var_name="reference_date", value_name="population")
population["year"] = population["reference_date"].str[-4:].astype(int)
population = population[population["year"].between(FIRST_YEAR_STATES, LAST_YEAR)]
population["state"] = population["state"].replace(STATE_NAMES_EN)
population = (population[["state", "year", "reference_date", "population"]]
              .sort_values(["year", "state"]).reset_index(drop=True))

# Check: 16 states per year, and the population of Germany as the sum of the states
population.groupby("year")["population"].agg(states="count", germany="sum")

,states,germany
year,,
2022,16,83118501
2023,16,83456045
2024,16,83577140
2025,16,83467117


Each year has 16 federal states. The sum of the states is between 83.1 and 83.6 million inhabitants in every year.

## 8. Right-Wing Offences per 100,000 Inhabitants

The state table is reshaped to one row per state and year and joined with the population. The rate is calculated as offences divided by inhabitants, multiplied by 100,000.

The rates printed in the Bundestag papers are not used. They are rounded to whole numbers and based on other population figures, so they would not be comparable across the four years.

In [12]:
pmk_states_rates = (pmk_states.pivot(index=["year", "state"], columns="offence", values="cases")
                    .reset_index().rename_axis(columns=None))
pmk_states_rates["state"] = pmk_states_rates["state"].replace(STATE_NAMES_EN)

pmk_states_rates = pmk_states_rates.merge(population[["state", "year", "population"]],
                                          on=["state", "year"], validate="one_to_one")
pmk_states_rates["total_per_100k"] = (100_000 * pmk_states_rates["total"] / pmk_states_rates["population"]).round(1)
pmk_states_rates["violent_per_100k"] = (100_000 * pmk_states_rates["violent"] / pmk_states_rates["population"]).round(2)

print("Rows:", len(pmk_states_rates), "| missing values:", int(pmk_states_rates.isna().sum().sum()))
pmk_states_rates[pmk_states_rates["year"] == LAST_YEAR].sort_values("total_per_100k", ascending=False)

Rows: 64 | missing values: 0


,year,state,total,violent,population,total_per_100k,violent_per_100k
55,2025,Mecklenburg-Western Pomerania,2282,103,1573685,145.0,6.55
61,2025,Saxony-Anhalt,3007,90,2120252,141.8,4.24
51,2025,Brandenburg,3557,145,2551889,139.4,5.68
63,2025,Thuringia,2414,127,2078946,116.1,6.11
60,2025,Saxony,3406,84,4026599,84.6,2.09
50,2025,Berlin,3021,133,3700577,81.6,3.59
53,2025,Hamburg,1517,120,1869473,81.1,6.42
52,2025,Bremen,557,12,706915,78.8,1.70
59,2025,Saarland,583,33,1007342,57.9,3.28
62,2025,Schleswig-Holstein,1438,58,2958519,48.6,1.96


The table has 64 rows (16 federal states, four years) and no missing values, so every state found its population figure.

As a plausibility check, the calculated rates of 2025 can be compared with the rounded rates printed in Bundestag paper 21/5639 (page 3). For the four states with the highest rates the paper gives 145, 142, 139 and 116. The values calculated here are 145.0, 141.8, 139.4 and 116.1.

## 9. Save the Clean Tables

Four tables are saved for the analysis. The column names are documented below.

| File | Rows | Columns |
|---|---|---|
| `afd_results.csv` | One per election and federal state (and Germany) | `election`, `year`, `state`, `afd_votes`, `afd_share_pct` |
| `population_states.csv` | One per federal state and year, 2022 to 2025 | `state`, `year`, `reference_date`, `population` |
| `pmk_right_federal.csv` | One per year and type of offence, 2014 to 2025 | `year`, `offence`, `cases`, `source` |
| `pmk_right_states.csv` | One per federal state and year, 2022 to 2025 | `year`, `state`, `total`, `violent`, `population`, `total_per_100k`, `violent_per_100k` |

The two transcribed tables `pmk_rechts_bund.csv` and `pmk_rechts_laender.csv` stay unchanged in `data/clean/`. They hold the source of every value.

In [13]:
afd_results.to_csv(DATA_CLEAN / "afd_results.csv", index=False)
population.to_csv(DATA_CLEAN / "population_states.csv", index=False)
pmk_federal.to_csv(DATA_CLEAN / "pmk_right_federal.csv", index=False)
pmk_states_rates.to_csv(DATA_CLEAN / "pmk_right_states.csv", index=False)

saved = ["afd_results.csv", "population_states.csv", "pmk_right_federal.csv", "pmk_right_states.csv"]
print("Saved to data/clean:", [name for name in saved if (DATA_CLEAN / name).exists()])

Saved to data/clean: ['afd_results.csv', 'population_states.csv', 'pmk_right_federal.csv', 'pmk_right_states.csv']


## 10. Limitations

**The crime data is not published in one place.** The figures are spread across fact sheets of the BKA and BMI, older reports and printed papers of the Bundestag (answers to parliamentary questions). Some older reports of the BMI could no longer be found online. All values were taken from the original documents; nothing was estimated or filled in. Whether the scattered publication is intended or follows from publication practice is not assessed here.

**Types of offence are missing for 2015 and 2016.** The reports of the BMI for these years were not available. Total and violent offences are available, taken from the ten-year series in the report for 2021.

**Coercion and threat is missing for 2020 and 2021.** The value for right-wing offences was not found in the available sources.

**Insult is a separate category from 2019 only.** Before that it is contained in "other offences".

**Totals per federal state are available from 2022 only.** For 2014 to 2021 no totals per state were found. The comparison of the federal states therefore covers four years.

**The state figures of 2022 and of 2023 to 2025 may not be fully comparable.** Bundestag paper 20/7594 gives the figures of 2022 by time of the offence ("Tatzeit: 2022"). Paper 21/1418 states that the annual figures of 2024 were used, and paper 21/5639 gives annual figures of 2024 and 2025 with the cut-off date 31 January of the following year. For the figures of 2023, printed in paper 21/1418, the paper names no cut-off date. Whether these methods lead to the same count was not verified. In all four years the sum of the states equals the federal figure of the annual report (Section 5).

**Violent offences per state in 2025 were counted.** Bundestag paper 21/5639 does not print this number per state. It lists the 1,598 violent offences one by one, each with the federal state (pages 5 to 60). The cases were counted per state. They are numbered from 1 to 1,598 without gaps, and their sum equals the federal figure. These 16 values are marked with `gezaehlt` in the transcribed table.

**The figures can still change.** The statistics count cases when the police report them, and cases can be reported later. The figures for 2025 are the least settled.

**The population basis is 31 December.** The Bundestag papers use other reference dates for their own rates (paper 21/5639: 30 September). The rates calculated here therefore differ slightly from the printed ones.

**The European election of 2014 has no published share in the file.** It was calculated from votes and valid votes.

## 11. Data Sources

**Crime data.** Published by the Federal Criminal Police Office (Bundeskriminalamt, BKA) and the Federal Ministry of the Interior (Bundesministerium des Innern, BMI), and by the German Bundestag as answers of the Federal Government to parliamentary questions. The source of every single value, with file name and page, is in the column `source` of the transcribed tables.

| Used for | Document | File in `data/raw` |
|---|---|---|
| Federal level 2014 | Bundestag paper 18/5758, pages 4, 5, 8 and 9 | `pmk_bund/1805758.pdf` |
| Federal level 2015 and 2016 (total, violent) | PMK figures 2021, ten-year series, pages 4 and 7 | `pmk_bund/2021PMKFallzahlen.pdf` |
| Federal level 2017 and 2018 | Offences by type 2017 and 2018, page 1 | `pmk_bund/pmk-2018-deliktsbereiche.pdf` |
| Federal level 2019 | Offences by type 2018 and 2019, page 1 | `pmk_bund/pmk-2019-deliktsbereiche.pdf` |
| Federal level 2020 | PMK figures 2020, pages 2 to 5 | `pmk_bund/2020PMKFallzahlen.pdf` |
| Federal level 2021 | PMK figures 2021, pages 4 to 7 | `pmk_bund/2021PMKFallzahlen.pdf` |
| Federal level 2022 | PMK fact sheets 2022, pages 4 to 7 | `pmk_bund/pmk2022-factsheets.pdf` |
| Federal level 2023 | PMK fact sheets 2023, pages 4 to 8 | `pmk_bund/pmk2023-factsheets.pdf` |
| Federal level 2024 | PMK figures 2024, pages 4 to 8 | `pmk_bund/2024PMKFallzahlen.pdf` |
| Federal level 2025 | PMK figures 2025, pages 4 to 9 | `pmk_bund/2025PMKFallzahlen.pdf` |
| Federal states 2022 | Bundestag paper 20/7594, page 8 | `pmk_laender/2007594.pdf` |
| Federal states 2023 and 2024 | Bundestag paper 21/1418, pages 2 to 4 | `pmk_laender/2101418.pdf` |
| Federal states 2025 | Bundestag paper 21/5639, page 2 and annex (pages 5 to 60) | `pmk_laender/2105639.pdf` |

Page numbers are the pages of the PDF file, not the printed page numbers. Three values were taken from the report of the following year, which repeats the figure of the year before: insult 2019, property damage 2020, and coercion and threat 2022.

**Election data.** Published by the Federal Returning Officer (Die Bundeswahlleiterin), Wiesbaden. Final results. The files of the elections 2017 to 2025 carry the licence note "Datenlizenz Deutschland – Namensnennung – Version 2.0". The file of the European election 2014 carries only the copyright note "(c) Der Bundeswahlleiter, Wiesbaden 2016".

| Used for | Election | File in `data/raw/wahlen` | Version |
|---|---|---|---|
| Section 6 | European election, 25 May 2014 | `ew_2014_2019_2024/ew2014_kerg.csv` | Final result |
| Section 6 | Federal election, 24 September 2017 | `btw17_kerg2.csv` | 6 October 2017 |
| Section 6 | European election, 26 May 2019 | `ew_2014_2019_2024/ew2019_kerg2.csv` | 24 June 2019 |
| Section 6 | Federal election, 26 September 2021 | `btw21_kerg2.csv` | 14 October 2021 |
| Section 6 | European election, 9 June 2024 | `ew_2014_2019_2024/ew2024_kerg2.csv` | 26 June 2024 |
| Section 6 | Federal election, 23 February 2025 | `btw25_kerg2.csv` | 14 March 2025 |

For the federal election of 2021 the original result is used, without the repeat election in parts of Berlin on 11 February 2024.

**Population data.** Federal Statistical Office (Statistisches Bundesamt, Destatis), table 12411-0010 "Bevölkerung: Bundesländer, Stichtag", retrieved on 5 October 2026. File: `data/raw/bevoelkerung/12411-0010_de.csv`.

The rates per 100,000 inhabitants and the AfD share in the European election of 2014 are own calculations based on these tables.